# 23. scikit-learn — `scikit-learn`
**Python Libraries Lab · Phase 2 · Machine Learning**

_Train, evaluate and use machine learning models with one consistent API._

✅ **Runs in Colab** as-is.

scikit learn is the standard toolkit for classic machine learning in Python. Every model, from a simple linear regression to a random forest, uses the same three steps. Create the model, fit it on training data, and predict on new data. Learn those once and you can try dozens of models.

**In this notebook:** ShopEasy wants to predict whether a customer will come back and buy again. You will create customer data, split it, train a logistic regression and a random forest, compare them with accuracy and a confusion matrix, see which features matter most, and wrap everything in a pipeline.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q scikit-learn
import importlib.metadata as md
for pkg in ['scikit-learn']:
    print(pkg, md.version(pkg))

## Step 1 · Make customer data
Each row is a customer. `returned` (1 or 0) is the label we want to predict.

In [ ]:
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
n = 2000
customers = pd.DataFrame({
    "orders_last_90d": rng.poisson(2, n),
    "avg_order_value": rng.normal(1500, 600, n).clip(100),
    "days_since_last": rng.integers(1, 120, n),
    "used_coupon": rng.integers(0, 2, n),
    "city": rng.choice(["Mumbai", "Delhi", "Pune", "Bengaluru"], n),
})
score = 0.6 * customers["orders_last_90d"] - 0.03 * customers["days_since_last"] + 0.8 * customers["used_coupon"] + rng.normal(0, 1, n)
customers["returned"] = (score > 0).astype(int)
print(customers["returned"].value_counts(normalize=True).round(2))
customers.head()

## Step 2 · Split into train and test
80% to learn from, 20% kept hidden for grading. `stratify` keeps the same share of 1s in both.

In [ ]:
from sklearn.model_selection import train_test_split

X = customers.drop(columns="returned")
y = customers["returned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

## Step 3 · A pipeline: prepare + model
Scale numbers, one-hot encode the city, then fit the model, all in one object that cannot leak test data.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

num_cols = ["orders_last_90d", "avg_order_value", "days_since_last", "used_coupon"]
prep = make_column_transformer((StandardScaler(), num_cols), (OneHotEncoder(), ["city"]))
logreg = make_pipeline(prep, LogisticRegression())
logreg.fit(X_train, y_train)
print("Test accuracy:", round(logreg.score(X_test, y_test), 3))
print("Prediction for first 5 test customers:", logreg.predict(X_test[:5]))
print("Probability of returning:", logreg.predict_proba(X_test[:5])[:, 1].round(2))

## Step 4 · Compare with a random forest
Same pattern, different model. Cross-validation gives a fairer score than one split.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

forest = make_pipeline(prep, RandomForestClassifier(n_estimators=200, random_state=42))
for name, model in [("Logistic regression", logreg), ("Random forest", forest)]:
    scores = cross_val_score(model, X_train, y_train, cv=5)
    print(f"{name:<20} CV accuracy {scores.mean():.3f} (+/- {scores.std():.3f})")
forest.fit(X_train, y_train)

## Step 5 · Evaluate properly
The confusion matrix shows the kinds of mistakes. Precision and recall matter when mistakes have different costs.

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

pred = forest.predict(X_test)
print(classification_report(y_test, pred, target_names=["won't return", "returns"]))
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["won't return", "returns"], cmap="Blues")
plt.show()

## Step 6 · Which features matter?
Permutation importance: shuffle one column and see how much the score drops.

In [ ]:
from sklearn.inspection import permutation_importance

imp = permutation_importance(forest, X_test, y_test, n_repeats=10, random_state=42)
pd.Series(imp.importances_mean, index=X_test.columns).sort_values().plot.barh(title="What predicts a returning customer?")
plt.show()

## ✍️ Your turn
Train a `DecisionTreeClassifier(max_depth=3, random_state=0)` inside a pipeline with `prep`, fit it on the training data, and store its **test accuracy** in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
answer = None  # your code here

In [ ]:
def check():
    from sklearn.tree import DecisionTreeClassifier
    exp = make_pipeline(prep, DecisionTreeClassifier(max_depth=3, random_state=0)).fit(X_train, y_train).score(X_test, y_test)
    ok = answer is not None and abs(answer - exp) < 1e-9
    return ok, f"Expected about {exp:.3f}, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
from sklearn.tree import DecisionTreeClassifier
tree = make_pipeline(prep, DecisionTreeClassifier(max_depth=3, random_state=0)).fit(X_train, y_train)
answer = tree.score(X_test, y_test)

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **fit / predict**
- **train_test_split**
- **Pipeline**
- **Cross-validation**
- **Confusion matrix**
- **Feature importance**

**Next:** 24. XGBoost (`xgboost`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/24_xgboost.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install scikit-learn
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.